In [4]:
import numpy as np
import math, time

# Schweinsberg-style "offspring" draw
# total = sum_i ((1 - p0) / U_i)^(1/alpha)
def schweinsberg_total(N=1000, alpha=1.1, p0=0.0, per_parent_floor=False, randomized_round=True, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    U = rng.random(N)  # Uniform(0,1)
    vals = ((1.0 - p0) / U) ** (1.0 / alpha)

    if per_parent_floor:
        # Original per-parent integerization
        return int(np.sum(np.floor(vals)))

    # Sum-then-round (like your second version)
    total = float(np.sum(vals))
    if randomized_round:
        int_part = math.floor(total)
        frac_part = total - int_part
        return int_part + (rng.random() < frac_part)
    else:
        return int(round(total))

# ---- 1) Constructive demonstration (guarantee > 1e8 once) ----
def constructive_demo(N=1000, alpha=1.1, p0=0.0, target=100_000_000, seed=123):
    rng = np.random.default_rng(seed)
    U = rng.random(N)
    # Choose one U so that its contribution alone exceeds `target`.
    # We need ((1-p0)/u)^(1/alpha) > target  =>  u < (1-p0)/target^alpha
    u_thresh = (1.0 - p0) / (target ** alpha)
    U[0] = u_thresh * 0.5  # safely below threshold -> contribution > target
    print(U[0])
    vals = ((1.0 - p0) / U) ** (1.0 / alpha)
    total = float(np.sum(vals))
    print(f"[Constructive] One tiny U makes sum explode: total={total:.3e} (N={N}, alpha={alpha})")
    return total

# ---- 2) Batched Monte Carlo search and tail-prob estimate ----
def mc_demo(N=1000, alpha=1.1, p0=0.0, target=100_000_000,
            per_parent_floor=False, randomized_round=True,
            batch=200_000, max_batches=100, seed=42):
    rng = np.random.default_rng(seed)
    start = time.time()

    exceed_count = 0
    total_draws = 0
    example_value = None

    for b in range(max_batches):
        # Vectorized: draw U once and reuse
        U = rng.random((batch, N))
        vals = ((1.0 - p0) / U) ** (1.0 / alpha)
        if per_parent_floor:
            totals = np.sum(np.floor(vals), axis=1)
        else:
            totals = np.sum(vals, axis=1)
            if randomized_round:
                int_parts = np.floor(totals).astype(np.int64)
                frac_parts = totals - int_parts
                totals = int_parts + (rng.random(batch) < frac_parts)

        totals = totals.astype(np.int64)
        total_draws += batch

        hits = totals >= target
        n_hits = int(np.sum(hits))
        exceed_count += n_hits

        if n_hits > 0 and example_value is None:
            example_value = int(totals[hits][0])

        # Print running estimate
        est = exceed_count / total_draws
        print(f"[Batch {b+1}/{max_batches}] draws={total_draws:,} hits={exceed_count}  "
              f"tail≈{est:.3e}")

        if n_hits > 0:
            print(f"  Example exceeding draw: {example_value:,}")
            break

    elapsed = time.time() - start
    if exceed_count == 0:
        print(f"No exceedance found after {total_draws:,} draws "
              f"(elapsed {elapsed:.1f}s). Estimated tail < {1/total_draws:.3e}")
    else:
        print(f"Found exceedance after {total_draws:,} draws "
              f"(elapsed {elapsed:.1f}s). Empirical tail ≈ {exceed_count/total_draws:.3e}")

# ---- Analytic back-of-envelope probability for the max term ----
def analytic_tail_prob(N=1000, alpha=1.1, p0=0.0, target=100_000_000):
    # P(one term > target) = (1-p0) / target^alpha
    p_single = (1.0 - p0) / (target ** alpha)
    # P(max > target) ≈ 1 - (1 - p_single)^N ≈ N * p_single for tiny p
    approx = 1 - (1 - p_single) ** N
    approx_lin = N * p_single
    print(f"[Analytic] p_single≈{p_single:.3e},  P(max>target)≈{approx:.3e} (linear≈{approx_lin:.3e})")

if __name__ == "__main__":
    N = 1000
    alpha = 1.1
    p0 = 0.0
    TARGET = 100_000_000

    print("=== 1) Constructive explosion ===")
    _ = constructive_demo(N=N, alpha=alpha, p0=p0, target=TARGET)

    print("\n=== 2) Analytic tail estimate ===")
    analytic_tail_prob(N=N, alpha=alpha, p0=p0, target=TARGET)

    print("\n=== 3) Monte Carlo search (may or may not hit, since it’s rare) ===")
    mc_demo(N=N, alpha=alpha, p0=p0, target=TARGET,
            per_parent_floor=False, randomized_round=True,
            batch=200_000, max_batches=100, seed=7)


=== 1) Constructive explosion ===
7.924465962305554e-10
[Constructive] One tiny U makes sum explode: total=1.878e+08 (N=1000, alpha=1.1)

=== 2) Analytic tail estimate ===
[Analytic] p_single≈1.585e-09,  P(max>target)≈1.585e-06 (linear≈1.585e-06)

=== 3) Monte Carlo search (may or may not hit, since it’s rare) ===


[Batch 1/100] draws=200,000 hits=1  tail≈5.000e-06
  Example exceeding draw: 438,797,482
Found exceedance after 200,000 draws (elapsed 4.8s). Empirical tail ≈ 5.000e-06


In [5]:
constructive_demo(N=1000, alpha=1.1, p0=0.0, target=100_000_000)

7.924465962305554e-10
[Constructive] One tiny U makes sum explode: total=1.878e+08 (N=1000, alpha=1.1)


187793747.4827014

In [ ]:
rng = np.random.default_rng(42)

In [28]:
U = rng.random(100_000_000)
U.min()

np.float64(6.5928036363516185e-09)

In [36]:
U = 0.00000001
vals = ((1.0 - 0) / U) ** (1.0 / 1.1)
vals

18738174.22860383